# S1-23 `hobo build` draft POC — HOBO-613

`hobo build` orchestrates codegen-freshness checks, `@hobo/componentize` for ts/js services,
and a declared **OCI fallback** for everything else, emitting a digest-stable build manifest.
The load-bearing invariant is **digest parity**: the artifact digest from the build pipeline
must equal the direct componentize path *and* the committed golden S0-07 fixture
(`test/fixtures/hello-bindle.digest.json`). Every run below builds a real fixture project.

In [1]:
import { strict as assert } from "node:assert";
import { readFileSync } from "node:fs";
import { mkdir, mkdtemp, readFile, rm, writeFile } from "node:fs/promises";
import { tmpdir } from "node:os";
import { dirname, join } from "node:path";
import { fileURLToPath } from "node:url";

import { componentizeHelloBindle } from "../../../../packages/componentize/src/index.ts";
import {
  executeCodegenRequest,
  generateCodegenArtifacts,
} from "../../technical-proofs/hobo-codegen-cli-contract/src/codegen-cli-contract.ts";
import { BuildError, createBuildCli, runBuild, writeBuildFixtureProject } from "./src/index.ts";

const GOLDEN = JSON.parse(
  readFileSync(new URL("./test/fixtures/hello-bindle.digest.json", import.meta.url), "utf8"),
);

async function makeProject() {
  const projectRoot = await mkdtemp(join(tmpdir(), "hobo-build-nb-"));
  await writeBuildFixtureProject(projectRoot);
  await executeCodegenRequest({ projectRoot, check: false, guestLanguage: "ts", engine: generateCodegenArtifacts });
  const target = join(projectRoot, "services/hello-bindle/src/service.ts");
  const source = fileURLToPath(
    new URL("../../../../packages/componentize/test/fixtures/hello-bindle/src/service.ts", import.meta.url),
  );
  await mkdir(dirname(target), { recursive: true });
  await writeFile(target, await readFile(source, "utf8"));
  return projectRoot;
}

const projectRoot = await makeProject();
try {
  const result = await runBuild({
    json: true, projectRoot, targetRuntime: "auto", sourceDateEpoch: GOLDEN.sourceDateEpoch,
  });
  for (const artifact of result.artifacts) {
    console.log(
      `artifact: service=${artifact.service} kind=${artifact.kind}` +
        (artifact.fallbackReason ? ` fallback="${artifact.fallbackReason}"` : ""),
    );
  }

  // Digest parity: build path === direct componentize path === golden fixture.
  const hello = result.artifacts.find((a) => a.service === "hello-bindle");
  const direct = await componentizeHelloBindle({
    sourceRoot: join(projectRoot, "services/hello-bindle"),
    sourceDateEpoch: GOLDEN.sourceDateEpoch,
  });
  assert.equal(hello.digest, direct.component.digest);
  assert.equal(hello.digest, GOLDEN.digest);
  console.log(`digest parity held: ${hello.digest}`);

  // Reproducibility: rebuilding the same tree yields the same manifest digest.
  const rebuild = await runBuild({
    json: true, projectRoot, targetRuntime: "auto", sourceDateEpoch: GOLDEN.sourceDateEpoch,
  });
  console.log(`manifest reproducible: ${rebuild.manifestDigest === result.manifestDigest}`);
} finally {
  await rm(projectRoot, { recursive: true, force: true });
}

artifact: service=hello-bindle kind=wasm-component
artifact: service=legacy-worker kind=oci-container fallback="unsupported-language: python"
digest parity held: sha256:c0d38866fc7ad727a9c2bf877919fb5aa0a3f8a5057f6bd59b55f38f8fe8dfc0
manifest reproducible: true


## CLI envelope and fail-closed codegen freshness

The CLI wraps the same `runBuild` and renders a stable JSON envelope; a generated tree that
drifts from what `hobo codegen` would produce fails the build with a typed `BuildError`.

In [2]:
import { strict as assert } from "node:assert";
import { readFileSync } from "node:fs";
import { mkdir, mkdtemp, readFile, rm, writeFile } from "node:fs/promises";
import { tmpdir } from "node:os";
import { dirname, join } from "node:path";
import { fileURLToPath } from "node:url";

import { componentizeHelloBindle } from "../../../../packages/componentize/src/index.ts";
import {
  executeCodegenRequest,
  generateCodegenArtifacts,
} from "../../technical-proofs/hobo-codegen-cli-contract/src/codegen-cli-contract.ts";
import { BuildError, createBuildCli, runBuild, writeBuildFixtureProject } from "./src/index.ts";

const GOLDEN = JSON.parse(
  readFileSync(new URL("./test/fixtures/hello-bindle.digest.json", import.meta.url), "utf8"),
);

async function makeProject() {
  const projectRoot = await mkdtemp(join(tmpdir(), "hobo-build-nb-"));
  await writeBuildFixtureProject(projectRoot);
  await executeCodegenRequest({ projectRoot, check: false, guestLanguage: "ts", engine: generateCodegenArtifacts });
  const target = join(projectRoot, "services/hello-bindle/src/service.ts");
  const source = fileURLToPath(
    new URL("../../../../packages/componentize/test/fixtures/hello-bindle/src/service.ts", import.meta.url),
  );
  await mkdir(dirname(target), { recursive: true });
  await writeFile(target, await readFile(source, "utf8"));
  return projectRoot;
}

const projectRoot = await makeProject();
try {
  const cli = await createBuildCli().run(["build", "--json"], {
    cwd: projectRoot,
    env: { ...process.env, SOURCE_DATE_EPOCH: GOLDEN.sourceDateEpoch },
  });
  const envelope = JSON.parse(cli.stdout);
  const hello = envelope.artifacts.find((a) => a.service === "hello-bindle");
  assert.equal(hello.digest, GOLDEN.digest);
  console.log(`cli envelope: ok=${envelope.ok} command=${envelope.command} artifacts=${envelope.artifacts.length}`);

  // Mutate the generated tree -> the next build fails closed with a drift error.
  const generatedManifest = join(projectRoot, ".hobo/generated/codegen-manifest.json");
  await writeFile(generatedManifest, `${await readFile(generatedManifest, "utf8")}\n// stale\n`);
  try {
    await runBuild({ json: true, projectRoot, targetRuntime: "auto", sourceDateEpoch: GOLDEN.sourceDateEpoch });
  } catch (error) {
    assert.ok(error instanceof BuildError);
    console.log(`stale generated tree rejected: code=${error.code}`);
  }
  console.log("CONTRACT HELD: hobo build is digest-stable, fallback-aware, and refuses stale codegen");
} finally {
  await rm(projectRoot, { recursive: true, force: true });
}

cli envelope: ok=true command=build artifacts=2
stale generated tree rejected: code=HOBO_BUILD_CODEGEN_DRIFT
CONTRACT HELD: hobo build is digest-stable, fallback-aware, and refuses stale codegen
